In [ ]:
import torch
import random

from data_utils import load_abc_dataset, train_val_split
from tokenizer_utils import train_tokenizer
from model import ABCTransformer

In [ ]:
# Data
MAX_EXAMPLES = 50000
VAL_RATIO = 0.1

# Tokenizer
VOCAB_SIZE = 1024

# Model
BLOCK_SIZE = 256
EMBED_DIM = 256
NUM_HEADS = 4
NUM_LAYERS = 4
DROPOUT = 0.1

# Training
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
MAX_STEPS = 3000
EVAL_INTERVAL = 200
EVAL_ITERS = 50

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
texts = load_abc_dataset(max_examples=MAX_EXAMPLES)
train_texts, val_texts = train_val_split(texts, val_ratio=VAL_RATIO)

print("Total:", len(texts))
print("Train:", len(train_texts))
print("Val:", len(val_texts))
print()
print(train_texts[0][:500])

In [ ]:
tokenizer = train_tokenizer(
    train_texts,
    vocab_size=VOCAB_SIZE,
    save_path="tokenizer.json"
)

print("Tokenizer vocab size:", tokenizer.get_vocab_size())

In [ ]:
train_token_ids = []
for text in train_texts:
    ids = tokenizer.encode(text).ids
    if len(ids) > 1:
        train_token_ids.extend(ids + [tokenizer.token_to_id("[EOS]")])

val_token_ids = []
for text in val_texts:
    ids = tokenizer.encode(text).ids
    if len(ids) > 1:
        val_token_ids.extend(ids + [tokenizer.token_to_id("[EOS]")])

train_data = torch.tensor(train_token_ids, dtype=torch.long)
val_data = torch.tensor(val_token_ids, dtype=torch.long)

print("Train tokens:", len(train_data))
print("Val tokens:", len(val_data))

In [ ]:
def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in ix])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [ ]:
xb, yb = get_batch("train")
print(xb.shape, yb.shape)
print(xb[0][:20])
print(yb[0][:20])

In [ ]:
vocab_size = tokenizer.get_vocab_size()

model = ABCTransformer(
    vocab_size=vocab_size,
    block_size=BLOCK_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

print("Parameters (M):", sum(p.numel() for p in model.parameters()) / 1e6)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}

    for split in ["train", "val"]:
        split_losses = []

        for _ in range(EVAL_ITERS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            split_losses.append(loss.item())

        losses[split] = sum(split_losses) / len(split_losses)

    model.train()
    return losses

In [ ]:
model.train()

for step in range(MAX_STEPS):
    if step % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
torch.save(model.state_dict(), "abc_transformer.pt")
print("Model saved.")

In [ ]:
@torch.no_grad()
def generate_text(prompt="", max_new_tokens=300, temperature=1.0):
    model.eval()

    if prompt.strip():
        start_ids = tokenizer.encode(prompt).ids
    else:
        bos_id = tokenizer.token_to_id("[BOS]")
        if bos_id is None:
            start_ids = [0]
        else:
            start_ids = [bos_id]

    x = torch.tensor([start_ids], dtype=torch.long, device=device)

    y = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature)
    token_ids = y[0].tolist()

    text = tokenizer.decode(token_ids)
    return text

In [ ]:
sample = generate_text(
    prompt="X:1\nT:Generated Tune\nM:4/4\nL:1/8\nK:D\n",
    max_new_tokens=400,
    temperature=0.9
)

print(sample)

In [ ]:
for i in range(3):
    print("=" * 80)
    print(f"SAMPLE {i+1}")
    print("=" * 80)
    print(generate_text(
        prompt="X:1\nT:Generated Tune\nM:4/4\nL:1/8\nK:G\n",
        max_new_tokens=300,
        temperature=0.95
    ))
    print()